# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Arslan
**Student ID:**
**Group:**
**Repository:** https://github.com/ars512/css4007-hw3

## AI tool disclosure

I used Claude Code (Claude Sonnet 5.5) to write the notebook code and the Part A working, and to draft the conceptual answers. Parts that depend on my own run outputs (B3 examples, C2 marks, C3 analysis, numbers in the final questions) are marked "fill in after running" and must be completed from the printed outputs.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

Tokenisation: lowercase, split on spaces and hyphens, drop punctuation, no stop words, no stemming.

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 = **8.333**

**Query terms:** how, does, rag, retrieve, information

**TF, DF and IDF of each query term** (N = 3): idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(7) = 1.9459 for every term, because n(t) = 0.

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document:**

> Every term has f(t,d) = 0, so every term contributes idf · 0 · 2.5 / (0 + …) = 0.
> - D1 = 0 + 0 + 0 + 0 + 0 = 0
> - D2 = 0 + 0 + 0 + 0 + 0 = 0
> - D3 = 0 + 0 + 0 + 0 + 0 = 0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 1 (tie) |
| D2 | 0.0000 | 1 (tie) |
| D3 | 0.0000 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl** = 25 / 3 = **8.333** (stemming does not change lengths)

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | ln((3 − 1 + 0.5)/(1 + 0.5)) = ln(1.6667) = 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document**, worked:

> Only `retriev` has f > 0, and only in D2. Length norm for D2: 1 − 0.75 + 0.75 · 10 / 8.333 = 0.25 + 0.9 = 1.15, so k1 · 1.15 = 1.725.
> - D2: 0.5108 · 2 · 2.5 / (2 + 1.725) = 2.5541 / 3.725 = **0.6857**
> - D1 = 0, D3 = 0 (no query term present)

| Document | BM25 Score | Rank |
|---|---:|---:|
| D2 | 0.6857 | 1 |
| D1 | 0.0000 | 2 (tie) |
| D3 | 0.0000 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **(a)** None. All three scores are 0, so there is no first document, only a three-way tie. This is not an arithmetic mistake: after tokenising, the query token `retrieve` never appears in any document. D2 has `retrieves` and `retrieval`, which are different strings for BM25, and `how`, `does`, `rag`, `information` appear nowhere. The check holds: BM25 matches strings, not meanings.
>
> **(b)** D2, with 0.6857. Stemming maps `retrieve`, `retrieves` and `retrieval` to `retriev`, so D2 matches the query (tf = 2). It is the only match, and its idf is small (0.51) because one of three documents contains it. The "obvious" terms `rag` and `information` still contribute nothing: D2 says "Retrieval-Augmented Generation", not "RAG".

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | √(0.49+0.01+0.16) = √0.66 = 0.8124 | 0.78 / (0.9644·0.8124) = 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | √(0.81+0.09+0.36) = √1.26 = 1.1225 | 1.08 / (0.9644·1.1225) = 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | √(0.01+0.64+0.04) = √0.69 = 0.8307 | 0.34 / (0.9644·0.8307) = 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D2 | 0.9977 | 1 |
| D1 | 0.9956 | 2 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 (0.9977), by a hair over D1 (0.9956). D3 is far away (0.4244).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Partly. Against A1 (b) the top document agrees (D2 first), but cosine also separates D1 from D3 (D2 > D1 > D3), whereas BM25 (b) leaves D1 and D3 tied at 0. Against A1 (a) it disagrees completely: BM25 as written found nothing (three-way tie), while cosine still ranks D2 first.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 needs the same token. The query says `retrieve`, `RAG`, `information`; D2 says `retrieves`, `Retrieval-Augmented Generation`, `documents`. No shared string (without stemming), so BM25 sees nothing. An embedding model can place "RAG" near "Retrieval-Augmented Generation" and "retrieve" near "retrieves"/"retrieval", so D2 lands close to the query vector. The flip side: D1 ("generate text and answer questions") also gets a very high cosine (0.9956) although it says nothing about retrieval, because embeddings are fuzzy; BM25 would never give it a score. (The vectors here are given, not computed, so this is an assumption about what a real model does.)

Check of the Part A arithmetic (the working above is by hand).

In [ ]:
import math, re
docs = ["Large language models can generate text and answer questions.",
        "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
        "Embeddings represent text as numerical vectors."]
query = "How does RAG retrieve information?"
stems = {'retrieve': 'retriev', 'retrieves': 'retriev', 'retrieval': 'retriev', 'generate': 'gener', 'generating': 'gener',
         'generation': 'gener', 'information': 'inform', 'does': 'doe'}
def tok(s, stem):
    t = re.findall(r"[a-z0-9]+", s.lower().replace('-', ' '))
    return [stems.get(x, x) for x in t] if stem else t
for stem in (False, True):
    D = [tok(d, stem) for d in docs]; Q = tok(query, stem); N = len(D); avgdl = sum(map(len, D)) / N
    scores = [0.0] * N
    for t in Q:
        n = sum(t in d for d in D); idf = math.log((N - n + .5) / (n + .5))
        for i, d in enumerate(D):
            f = d.count(t)
            if f: scores[i] += idf * f * 2.5 / (f + 1.5 * (.25 + .75 * len(d) / avgdl))
    print("stemming" if stem else "as written", "| lengths", [len(d) for d in D], "avgdl", round(avgdl, 3), "| scores", [round(s, 4) for s in scores])
qv = [.8, .2, .5]
for name, v in [("D1", [.7, .1, .4]), ("D2", [.9, .3, .6]), ("D3", [.1, .8, .2])]:
    dot = sum(a * b for a, b in zip(qv, v)); mq = math.sqrt(sum(a * a for a in qv)); mdv = math.sqrt(sum(a * a for a in v))
    print(name, "dot", round(dot, 4), "|q|", round(mq, 4), "|d|", round(mdv, 4), "cos", round(dot / (mq * mdv), 4))

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [ ]:
import os, json, re, warnings
import numpy as np
from dotenv import load_dotenv
load_dotenv()
warnings.filterwarnings("ignore")

corpus = [json.loads(l) for l in open("data/corpus.jsonl", encoding="utf8")]
ids = [c["id"] for c in corpus]
texts = [c["text"] for c in corpus]
byid = {c["id"]: c for c in corpus}
queries = json.load(open("data/queries.json", encoding="utf8"))["queries"]
qa = json.load(open("data/qa.json", encoding="utf8"))["questions"]
print(len(corpus), "chunks;", len(queries), "queries;", len(qa), "questions")

## B1 — BM25 with `rank_bm25`

**My tokenisation:** lowercase, `re.findall(r"\w+", text.lower())` (Unicode-aware, so Kazakh and Russian words survive; `GR-305` becomes the two tokens `gr` and `305`), **no stemming, no stop-word removal**. `BM25Okapi(k1=1.5, b=0.75)`. Consequences: no morphology (`retrieves` ≠ `retrieve`), and a Kazakh query can only match Kazakh chunks, because it shares no word with the English ones.

In [ ]:
from rank_bm25 import BM25Okapi
def tokenize(s): return re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def first_line(c, n=80):
    return c["text"].split("\n")[0][:n]
def bm25_top(q, k=3):
    s = bm25.get_scores(tokenize(q)); o = np.argsort(-s)[:k]
    return [(ids[i], float(s[i])) for i in o]

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("google/embeddinggemma-300m", token=os.getenv("HF_TOKEN"))   # default float32
doc_emb = model.encode_document(texts, show_progress_bar=False)   # computed once, kept in a numpy array
print("embedding matrix:", doc_emb.shape)
def emb_top(q, k=3):
    s = model.similarity(model.encode_query(q), doc_emb)[0].numpy(); o = np.argsort(-s)[:k]
    return [(ids[i], float(s[i])) for i in o]

## My own queries

Written, with their `relevant` ids, **before** running them.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which form is the household income declaration, GR-210? | GO-07, GO-03, GO-10, GO-13 | **BM25.** The rare exact token `210` (with `gr`) occurs in only a few chunks, so BM25 gives it a high idf; an embedding model tends to blur the form codes GR-104/GR-210/GR-305 into "a grant form". |
| Q-10 | Can I take a year off university and keep my money? | GO-18 | **Embeddings.** The handbook says "academic leave" and "payments stop"; the query says "year off" and "keep my money". Only a paraphrase, almost no shared words. |

In [ ]:
my_queries = [
    {"id": "Q-09", "text": "Which form is the household income declaration, GR-210?", "relevant": ["GO-07", "GO-03", "GO-10", "GO-13"]},
    {"id": "Q-10", "text": "Can I take a year off university and keep my money?", "relevant": ["GO-18"]},
]
all_queries = queries + my_queries

## Results for all ten or more queries: top 3 from each retriever

In [ ]:
results = {}
for q in all_queries:
    b, e = bm25_top(q["text"]), emb_top(q["text"])
    rel = set(q["relevant"])
    results[q["id"]] = {"bm25": b, "emb": e, "bm25_hit": any(i in rel for i, _ in b), "emb_hit": any(i in rel for i, _ in e)}
    print(f"\n=== {q['id']}: {q['text']}\n    relevant: {q['relevant']}")
    for name, top in (("BM25", b), ("EmbeddingGemma", e)):
        print(f"  {name}  hit@3 = {any(i in rel for i, _ in top)}")
        for r, (i, s) in enumerate(top, 1):
            print(f"    {r}. {i}  {s:8.4f}  {'*' if i in rel else ' '} {first_line(byid[i])}")

## B3 — comparison

The table is printed by the next cell from the actual runs (not typed by hand).

In [ ]:
print(f"{'Query':6} | {'BM25 top 3':26} | hit | {'Embedding top 3':26} | hit")
for q in all_queries:
    r = results[q["id"]]
    print(f"{q['id']:6} | {', '.join(i for i, _ in r['bm25']):26} | {'Y' if r['bm25_hit'] else 'N':3} | {', '.join(i for i, _ in r['emb']):26} | {'Y' if r['emb_hit'] else 'N'}")
print("hit@3 total: BM25", sum(r["bm25_hit"] for r in results.values()), "/", len(results),
      "| Embedding", sum(r["emb_hit"] for r in results.values()), "/", len(results))

**Both retrievers agree:** candidate Q-05 (Astana / Nur-Sultan): the query contains the distinctive words `Astana` and `Nur-Sultan`, which sit in one chunk (KZ-05), so a lexical match and a semantic match point to the same place. Confirm against the printed top 3. _Fill in after running the notebook (see the cell outputs above)._

**BM25 did better:** candidates to check: Q-01 (`GR-305`: the exact token `305` versus three near-identical form names) and Q-09 (`GR-210`). Name the exact string BM25 matched that the embedding model blurred. _Fill in after running the notebook (see the cell outputs above)._

**Embeddings did better:** candidates to check: Q-02 ("poor", "financial help" vs the handbook's "low-income", "grant"), Q-03 (Kazakh question over mostly English chunks), Q-10 ("year off" vs "academic leave"). Name what BM25 could not match (synonym, paraphrase or language). _Fill in after running the notebook (see the cell outputs above)._

Q-04: look at what came first and read that chunk's title. GO-05 is titled "Archived: the 2025 scheme (no longer in force)" and contains "minimum GPA" literally. _Fill in after running the notebook (see the cell outputs above)._

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [ ]:
import tiktoken
from openai import OpenAI
client = OpenAI()   # reads OPENAI_API_KEY from the environment (.env); the key is never printed
MODEL = "gpt-5.6-luna"          # the model required by the assignment
FALLBACK_MODEL = "gpt-4o-mini"   # used only if the account has no access to MODEL (see the printed notice)
enc = tiktoken.get_encoding("o200k_base")
ntok = lambda s: len(enc.encode(s))

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def context_block(chunk_ids):
    return "\n\n".join(f"[{n}] {byid[i]['title']}\n{byid[i]['text']}" for n, i in enumerate(chunk_ids, 1))
def build_messages(question, chunk_ids=None):
    if chunk_ids is None:     # LLM only: the bare question, no context
        return [{"role": "user", "content": question}]
    user = f"Context:\n{context_block(chunk_ids)}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
def ask(messages):
    global MODEL
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages)
    except Exception as e:
        if MODEL == FALLBACK_MODEL or "model" not in str(e).lower():
            raise
        print(f"NOTICE: {MODEL} is not available for this key ({type(e).__name__}); using {FALLBACK_MODEL} for all calls from now on")
        MODEL = FALLBACK_MODEL
        r = client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content, r.usage.prompt_tokens
qa_by = {q["id"]: q for q in qa}

## C1 — the complete prompt, printed before it is sent

Question used: **C-01** (income band 2 amount and payment dates), retrieved with EmbeddingGemma, top 3.

In [ ]:
q = qa_by["C-01"]
top_ids = [i for i, _ in emb_top(q["question"], 3)]
msgs = build_messages(q["question"], top_ids)
for m in msgs:
    print(f"----- role: {m['role']} -----\n{m['content']}\n")

ctx = context_block(top_ids)
t_sys, t_q, t_ctx = ntok(SYSTEM), ntok(q["question"]), ntok(ctx)
t_all = ntok(msgs[0]["content"]) + ntok(msgs[1]["content"])
ans, usage_tokens = ask(msgs)
print("retrieved chunk ids:", top_ids)
print("system:", t_sys, "| question:", t_q, "| retrieved context:", t_ctx, "| sum of the three parts:", t_sys + t_q + t_ctx)
print("tiktoken total of both messages (incl. my 'Context:' / 'Question:' labels):", t_all)
print("usage.prompt_tokens:", usage_tokens)
print("chunks:", len(top_ids), "| tokens per chunk:", [ntok(byid[i]['text']) for i in top_ids],
      "| context size:", t_ctx, "tokens,", len(ctx), "characters")
print("\nANSWER:", ans)

The numbers are printed by the cell above; copy them into this table after the run:

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | see output |
| Question | see output |
| Retrieved context | see output |
| **Total input** | see output |
| **`usage.prompt_tokens` from the response** | see output |

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> The API wraps every message in chat-format special tokens (role markers, delimiters) and primes the reply; none of that is in the raw text tiktoken counts. My own labels ("Context:", "Question:") are in the total but not in the three separate parts. The server-side template or tokenizer may also differ slightly from plain `o200k_base`, as seen in HW1. So `usage.prompt_tokens` is somewhat larger than the tiktoken sum.

**What exactly does retrieval add to the model's input?**

> Only text, literally: the three retrieved chunks pasted into the user message as numbered blocks (title + text), a few hundred tokens plus labels. No document, no index, no search tool, no vectors. The model reads those characters as ordinary prompt tokens and continues them.

## C2 — three systems, six questions

In [ ]:
c2 = {}
for q in qa:
    b_ids = [i for i, _ in bm25_top(q["question"], 3)]
    e_ids = [i for i, _ in emb_top(q["question"], 3)]
    c2[q["id"]] = {
        "llm": ask(build_messages(q["question"]))[0],
        "bm25": ask(build_messages(q["question"], b_ids))[0], "bm25_ids": b_ids,
        "emb": ask(build_messages(q["question"], e_ids))[0], "emb_ids": e_ids,
    }
    r = c2[q["id"]]
    print(f"\n=== {q['id']}: {q['question']}\nREFERENCE: {q['reference']}")
    print(f"--- LLM only:\n{r['llm']}\n--- BM25 RAG {b_ids}:\n{r['bm25']}\n--- Embedding RAG {e_ids}:\n{r['emb']}")

Mark each answer ✓ (agrees with `reference`) or ✗ by reading the printed answers above. Expectations before the run, to be replaced by real marks: C-04 and C-05 are Wikipedia facts, so even LLM-only may get them right; C-01 and C-02 need the handbook; C-03 may fail by retrieving the archived GO-05 (3.0); C-06 is correct only if the answer is "I do not know".

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | _mark_ | _mark_ | _mark_ |
| C-02 | _mark_ | _mark_ | _mark_ |
| C-03 | _mark_ | _mark_ | _mark_ |
| C-04 | _mark_ | _mark_ | _mark_ |
| C-05 | _mark_ | _mark_ | _mark_ |
| C-06 | _mark_ | _mark_ | _mark_ |
| **agrees with `reference`** | /6 | /6 | /6 |

## C3 — question 1 of 3: C-03

Why chosen: the archived GO-05 states `minimum GPA ... 3.0` in the same words as the question; check which retriever put it above GO-02 and whether the model cited it.

Sections 1–3 (retrieved ids and scores, the context blocks) are printed by the next cell. Sections 4–5 are judged from that output.

**4. How retrieval changed the answer** (more accurate, more specific, more grounded, or wrong because the wrong chunks came back): _Fill in after running the notebook (see the cell outputs above)._

**5. What was missed, and was it a retrieval problem or a generation problem?** Test: fact in the context but missing from the answer means generation failed; fact not in the context means retrieval failed. The cell prints which `relevant` chunks were in each context; quote them. _Fill in after running the notebook (see the cell outputs above)._

In [ ]:
def show(qid):
    q = qa_by[qid]; r = c2[qid]
    print(q["question"], "\nREFERENCE:", q["reference"], "\nrelevant:", q["relevant"])
    print("\n1. BM25 top 3:", [(i, round(s, 4)) for i, s in bm25_top(q["question"])])
    print("2. EmbeddingGemma top 3:", [(i, round(s, 4)) for i, s in emb_top(q["question"])])
    print("\n3. Context sent to the model (BM25 RAG):\n" + context_block(r["bm25_ids"]))
    print("\n   Context sent to the model (Embedding RAG):\n" + context_block(r["emb_ids"]))
    print("\nLLM only:", r["llm"], "\n\nBM25 RAG:", r["bm25"], "\n\nEmbedding RAG:", r["emb"])
    for name in ("bm25_ids", "emb_ids"):
        print(f"{name}: relevant chunks in context ->", [i for i in q["relevant"] if i in r[name]],
              "| missing ->", [i for i in q["relevant"] if i not in r[name]])
show("C-03")

## C3 — question 2 of 3: C-06

Why chosen: no answer in the corpus (GO-20 only says master's students have a separate scheme). The only correct answer is that the system does not know; check whether either RAG system invented an amount.

Sections 1–3 (retrieved ids and scores, the context blocks) are printed by the next cell. Sections 4–5 are judged from that output.

**4. How retrieval changed the answer** (more accurate, more specific, more grounded, or wrong because the wrong chunks came back): _Fill in after running the notebook (see the cell outputs above)._

**5. What was missed, and was it a retrieval problem or a generation problem?** Test: fact in the context but missing from the answer means generation failed; fact not in the context means retrieval failed. The cell prints which `relevant` chunks were in each context; quote them. _Fill in after running the notebook (see the cell outputs above)._

In [ ]:
show("C-06")

## C3 — question 3 of 3: C-02

Why chosen: the answer needs two facts (14 calendar days and form GR-305) spread over GO-08, GO-12 and GO-23 (the last in Russian), and the corpus has three similar form names.

Sections 1–3 (retrieved ids and scores, the context blocks) are printed by the next cell. Sections 4–5 are judged from that output.

**4. How retrieval changed the answer** (more accurate, more specific, more grounded, or wrong because the wrong chunks came back): _Fill in after running the notebook (see the cell outputs above)._

**5. What was missed, and was it a retrieval problem or a generation problem?** Test: fact in the context but missing from the answer means generation failed; fact not in the context means retrieval failed. The cell prints which `relevant` chunks were in each context; quote them. _Fill in after running the notebook (see the cell outputs above)._

In [ ]:
show("C-02")

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 scores by overlap of exact tokens (term frequency, rarity via idf, length normalisation); embeddings map text to dense vectors and score by closeness in meaning, so they work without shared words. BM25 is lexical, embeddings are semantic.

**2. When can BM25 perform better than embeddings?**

> When the query hinges on an exact, rare string: a form code like `GR-305` or `GR-210`, an id, a name, a number. BM25 gives that token a high idf, while the embedding model tends to blur near-identical codes. It also needs no model at all.

**3. When can embeddings perform better than BM25?**

> When query and answer use different words for the same thing: synonyms ("poor" vs "low-income", "year off" vs "academic leave"), paraphrases, or another language (the Kazakh query Q-03 over mostly English chunks). BM25 matches nothing in those cases.

**4. Why is cosine similarity used with embeddings?**

> It compares the direction of two vectors and ignores their length, so a long and a short text about the same thing still score as similar. With normalised vectors it is just a dot product, which is cheap to compute against every chunk.

**5. What information is actually sent to an LLM in a RAG system?**

> A prompt made of text: the instructions, the retrieved chunks (as numbered blocks) and the question (see the printed C1 prompt). Nothing else.

**6. Does the LLM search the original documents itself?**

> No. The code searches (BM25 or embeddings) and pastes the winners into the prompt; the model never opens a document and can only use what was pasted, plus what it learned in training.

**7. What happens if retrieval returns an irrelevant chunk?**

> The model cannot know it is irrelevant except by reading it. Best case it ignores it or says "I do not know"; worst case it builds a confident, cited answer from the wrong text (for example the archived 3.0 GPA in GO-05 for C-03). See the C3 analysis for what happened in my runs.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> A bigger K raises the chance that the needed chunk is included (more recall), which helps when the answer is spread over several chunks (C-02 needs GO-08, GO-12 and GO-23). But every extra chunk adds tokens and noise; distractors (the archived rule, similar form names) can mislead the model or bury the right fact. The next cell runs C-02 with K = 1, 3 and 8. _Fill in after running the notebook (see the cell outputs above)._

In [ ]:
q = qa_by["C-02"]
for k in (1, 3, 8):
    top = [i for i, _ in emb_top(q["question"], k)]
    a, used = ask(build_messages(q["question"], top))
    print(f"--- K={k}  chunks={top}  relevant in context={[i for i in q['relevant'] if i in top]}  prompt_tokens={used}\n{a}\n")

**9. Is a vector database required to build a RAG system?**

> No. My notebook kept the vectors in a plain numpy array in memory (`doc_emb`, 74 × 768) and searched it by computing `model.similarity` against every row. A vector database only matters at scale (millions of chunks, persistence, approximate nearest-neighbour search).

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> hit@3 over the ten queries is printed by the B3 table cell; the C2 agreement counts come from the marks in the C2 table. _Fill in after running the notebook (see the cell outputs above)._

---
# Conclusion

> Next I would build hybrid retrieval (BM25 for exact codes, embeddings for paraphrases and other languages, merged with reciprocal rank fusion), because the two fail in opposite places in this corpus. I would also mark archived chunks in metadata and filter them out, and keep the prompt rule that the model must say "I do not know" when the context lacks the answer, because those are the failures that retrieval alone does not fix. Final figures should be quoted from my own run outputs.